In [ ]:
import numpy as np
import pandas as pd

In [ ]:
VAT = 0.19               # VAT for NR/Unit Calculation
VILC_GR = 0.0378         # VILC Annual Growth Rate

In [ ]:
# READING IN ELASTICITY AND REFRENCE DATAFRAMES
df_sim = pd.read_csv(r'C:/Users/40107922/Downloads/r2/simulation_input.csv')

own_to_own_elasticity_df = pd.read_csv(r'C:/Users/40107922/Downloads/r2/elasticity.csv')
reference_df = pd.read_csv(r'C:/Users/40107922/Downloads/r2/reference_abi_sellin-vol_pl-ptc.csv')
seg_mapping = pd.read_csv(r'C:/Users/40107922/Downloads/r2/segment_mapping.csv')

In [ ]:
valid_year_months = reference_df['year_month'].unique()

def adjust_year_month(ym, valid_list):
    if ym in valid_list:
        return ym, False
    else:
        year, month = ym.split('-')
        adjusted_ym = f"{int(year)-1}-{month}"
        if adjusted_ym in valid_list:
            return adjusted_ym, True
        else:
            # if neither original nor adjusted is valid, keep original and mark not converted
            return ym, False

# Apply function to dataframe
df_sim[['corrected_year_month', 'converted_flag']] = df_sim['year_month'].apply(
    lambda x: pd.Series(adjust_year_month(x, valid_year_months))
)

df_sim.rename(columns={'year_month': 'year_month_og'}, inplace = True)
df_sim.rename(columns={'corrected_year_month': 'year_month'}, inplace = True)

In [ ]:
df_sim_merge = pd.merge(df_sim, reference_df[['year_month', 'sku', 'reference_volume', 'reference_price_unit', 'markup', 'discount',
                                             'excise', 'vilc']], on = ['year_month', 'sku'], how = 'left')

df_sim_merge['reference_volume'] = df_sim_merge['reference_volume'].fillna(0)
df_sim_merge['reference_price_unit'] = df_sim_merge['reference_price_unit'].fillna(1)
df_sim_merge['markup'] = df_sim_merge['markup'].fillna(0)
df_sim_merge['discount'] = df_sim_merge['discount'].fillna(0)
df_sim_merge['excise'] = df_sim_merge['excise'].fillna(0)
df_sim_merge['vilc'] = df_sim_merge['vilc'].fillna(0)

df_sim_merge.loc[df_sim_merge['converted_flag'], 'vilc'] = df_sim_merge['vilc'] * (1+ VILC_GR)
df_sim_merge = pd.merge(df_sim_merge, seg_mapping, on = ['sku'], how = 'left')

df_sim_merge['reference_price_liter'] = df_sim_merge['reference_price_unit'] * 1000 / df_sim_merge['capacity']
df_sim_merge['reco_price_liter'] = df_sim_merge['price_per_unit'] * 1000 / df_sim_merge['capacity']

In [ ]:
df_sim_merge.head()

In [ ]:
sku_set = df_sim_merge['sku'].unique()

own_to_own_elasticity_df = own_to_own_elasticity_df[own_to_own_elasticity_df['target_sku'].isin(sku_set)]
own_to_own_elasticity_df = own_to_own_elasticity_df[own_to_own_elasticity_df['other_sku'].isin(sku_set)]


df_sim_merge = df_sim_merge[['year_month_og', 'year_month', 'sku', 'price_per_unit', 'reference_volume', 'reference_price_unit', 'markup', 'discount', 'excise', 'vilc']]

# Extract unique SKUs from your elasticity dfs or existing reference dfs
own_skus = own_to_own_elasticity_df['target_sku'].unique()
own_skus.sort()

# Extract all year_months from your data or define explicitly
all_months = df_sim_merge['year_month'].unique()
# If your year_month is string, you might want to convert to datetime for sorting:
all_months_date = pd.to_datetime(all_months)
all_months_date = pd.Series(all_months_date).sort_values().unique()


# Build full MultiIndex for own products x all months
full_index_own = pd.MultiIndex.from_product(
    [all_months_date, own_skus],
    names=['year_month_date', 'sku']
)

# Convert reference_df_own['year_month'] to datetime for merging
df_sim_merge['year_month_date'] = pd.to_datetime(df_sim_merge['year_month'])

# Set index to year_month and sku for easier reindexing
df_sim_merge = df_sim_merge.set_index(['year_month_date', 'sku'])

# Reindex to full index
df_sim_merge_padded = df_sim_merge.reindex(full_index_own)

df_sim_merge_padded['present'] = 'present'
df_sim_merge_padded.loc[df_sim_merge_padded['reference_volume'].isna(), 'present'] = 'missing'

# Fill missing volumes with zero
df_sim_merge_padded['reference_volume'] = df_sim_merge_padded['reference_volume'].fillna(0)

# Fill missing prices with the mean price per SKU (computed from existing data)
mean_prices = df_sim_merge['reference_price_unit'].mean()
df_sim_merge_padded['reference_price_unit'] =  df_sim_merge_padded['reference_price_unit'].fillna(mean_prices)
mean_prices2 = df_sim_merge['price_per_unit'].mean()
df_sim_merge_padded['price_per_unit'] =  df_sim_merge_padded['price_per_unit'].fillna(mean_prices2)

# Fill other numeric columns similarly or with zeros
for col in ['markup', 'discount', 'excise', 'vilc']:  # add columns as needed
    if col in df_sim_merge_padded.columns:
        df_sim_merge_padded[col] = df_sim_merge_padded[col].fillna(0)

# Reset index if you want back to columns
df_sim_merge_padded = df_sim_merge_padded.reset_index()

year_month_mapping = df_sim_merge.reset_index()
year_month_mapping = year_month_mapping[['year_month_date', 'year_month']].drop_duplicates()

df_sim_merge_padded = pd.merge(df_sim_merge_padded, year_month_mapping, how = 'left', on = 'year_month_date')

df_sim_merge_padded = df_sim_merge_padded.drop(columns = {'year_month_date', 'year_month_x'})
df_sim_merge_padded.rename(columns = {'year_month_y': 'year_month'}, inplace = True)

df_sim_merge_padded = pd.merge(df_sim_merge_padded, seg_mapping, on = ['sku'], how = 'left')
df_sim_merge_padded['reference_price_liter'] = df_sim_merge_padded['reference_price_unit'] * 1000 / df_sim_merge_padded['capacity']
df_sim_merge_padded['reco_price_liter'] = df_sim_merge_padded['price_per_unit'] * 1000 / df_sim_merge_padded['capacity']

df_sim_merge = df_sim_merge_padded.copy()

In [ ]:
# Prepare Elasticity Matrix
own_products = df_sim_merge['sku'].unique()

months = sorted(df_sim_merge['year_month'].unique())
num_months = len(months)
num_own = len(own_products)
# num_comp = len(competitor_products)

# For convenience, convert these to numpy arrays and keep track of product order
own_index = {p: i for i, p in enumerate(own_products)}
# competitor_index = {p: i for i, p in enumerate(competitor_products)}


E_price_to_volume = np.zeros((num_own, num_own))  # shape (price_changer, volume_changer)

for _, row in own_to_own_elasticity_df.iterrows():
    i = own_index[row['target_sku']]  # price changer
    j = own_index[row['other_sku']]   # volume changer
    E_price_to_volume[i, j] = row['elasticity']

In [ ]:
# Helper function to get reference arrays per month
def get_reference_arrays_own_sim(month, ref_df, product_list):
    df_m = ref_df[(ref_df['year_month'] == month) & (ref_df['sku'].isin(product_list))].set_index('sku').reindex(product_list)
    # Fill missing with zeros or appropriate defaults
    #df_m = df_m.fillna(0)
    volume = df_m['reference_volume'].values
    price = df_m['reference_price_liter'].values
    capacity = df_m['capacity'].values
    markup = df_m['markup'].values
    discount = df_m['discount'].values
    excise = df_m['excise'].values
    vilc = df_m['vilc'].values
    return volume, price, capacity, markup, discount, excise, vilc


# Volume calculation given price changes and elasticities
def calc_volume_sim(opt_price_unit, ref_volume_own,  ref_price_liter, capacity, elasticity_matrix_own):
    opt_price_liter = opt_price_unit * 1000 / capacity
    price_ratio_own = np.log(opt_price_liter / ref_price_liter)
    Q_own_opt = ref_volume_own * np.exp(elasticity_matrix_own.T @ price_ratio_own)
    return Q_own_opt


# MACO and NR Calculation based on volume and costs
def calc_MACO(opt_price_unit, opt_volume, capacity, markup, discount, excise, vilc):
    sales_units = (opt_volume * 100000) / capacity
    discount_pct = discount 
    excise_pct = excise      
    NR_per_unit = ((opt_price_unit / (1 + markup)) * (1 + discount_pct + excise_pct)) / (1 + VAT)
    NR = NR_per_unit * sales_units
    MACO = NR + vilc
    return NR, MACO, sales_units

def simulate(P_opt):
    prices_by_month = P_opt.reshape(num_months, num_own)

    sku_all = []
    year_month_all = []
    price_unit_opt_all = []
    volume_opt_all = []
    NR_opt_all = []
    MACO_opt_all = []

    for i, month in enumerate(months):
        # Get reference arrays for own products and competitors for this month
        (ref_vol_own, ref_price_own, capacity_own,
         markup_own, discount_own, excise_own, vilc_own) = get_reference_arrays_own_sim(month, df_sim_merge, own_products)
    

        price_opt_own = prices_by_month[i, :]
        price_opt_liter = price_opt_own * 1000 / capacity_own
        price_ref_unit = ref_price_own * capacity_own / 1000

        # Calculate own volumes
        vol_own_opt = calc_volume_sim(price_opt_own, ref_vol_own, ref_price_own, capacity_own, E_price_to_volume)

        
        # Calculate NR and MACO for own products for this month
        NR_own, MACO_own, _ = calc_MACO(price_opt_own, vol_own_opt, capacity_own,
                                       markup_own, discount_own, excise_own, vilc_own)

        sku_all.extend(own_products)
        year_month_all.extend([month] * len(own_products))
        price_unit_opt_all.extend(price_opt_own)
        volume_opt_all.extend(vol_own_opt)
        NR_opt_all.extend(NR_own)
        MACO_opt_all.extend(MACO_own)
        
    sim_output_df = pd.DataFrame({
        'sku': sku_all,
        'year_month': year_month_all,
        'price_per_unit': price_unit_opt_all,
        'volume_simulated': volume_opt_all,
        'NR_simulated': NR_opt_all,
        'MACO_simulated': MACO_opt_all
    })
    return sim_output_df
    

In [ ]:
simulation_output_df = simulate(df_sim_merge['price_per_unit'].values)
simulation_output_df = pd.merge(simulation_output_df, df_sim_merge[['year_month_og', 'year_month', 'sku', 'present']], on = ['year_month', 'sku'], 
                               how = 'left')

In [ ]:
simulation_output_df.head()

In [ ]:
test = pd.merge(simulation_output_df, df_sim_merge[['year_month', 'sku', 'reference_volume']], on = ['year_month', 'sku'], how = 'left')

In [ ]:
print(test['volume_simulated'].sum() - test['reference_volume'].sum())
test['volume_simulated'] - test['reference_volume']

In [ ]:
simulation_output_df[simulation_output_df['present']=='present']#.to_excel(r'Round 2/Final Input Output/simulation/simulation_output_trial1.xlsx', index=False)